# 05 - EDA Avancada: STJ Integras com Plotly

**Objetivo:** Análise exploratória completa em perspectiva de data scientist senior.

Cobrir: volume temporal, composição documental, ministros, origem processual (CNJ), assuntos enriquecidos, tempos processuais, qualidade de dados e insights.

## 1. Setup

In [ ]:
import json
import re
import sys
from pathlib import Path
from collections import Counter
from typing import Optional

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots

# Configurar renderer para exibir gráficos inline em qualquer ambiente
# "notebook_connected" usa CDN (mais leve); "notebook" inclui plotly.js embarcado
pio.renderers.default = "notebook_connected+plotly_mimetype+colab"
print(f'Plotly renderer: {pio.renderers.default}')

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    MOUNT_POINT = Path('/content/drive')
    if (MOUNT_POINT / 'MyDrive').exists():
        print('Google Drive já está montado.')
    else:
        drive.mount(str(MOUNT_POINT), force_remount=True)
    DRIVE_DATA = MOUNT_POINT / 'MyDrive/Mestrado/2026/llms/data'
else:
    DRIVE_DATA = Path.cwd() if Path.cwd().name == 'datajud_probe' else Path.cwd().parent
    if not (DRIVE_DATA / 'data').exists():
        DRIVE_DATA = DRIVE_DATA / 'data'

METADATA_RAW = DRIVE_DATA / 'raw' / 'stj_integras_metadata'
REPORTS_DATA = DRIVE_DATA / 'reports' / 'summaries'
FIGURES_DATA = DRIVE_DATA / 'reports' / 'figures'
REFERENCE_DATA = DRIVE_DATA / 'reference'

for path in [METADATA_RAW, REPORTS_DATA, FIGURES_DATA, REFERENCE_DATA]:
    path.mkdir(parents=True, exist_ok=True)

print(f'METADATA_RAW: {METADATA_RAW}')
print(f'REPORTS_DATA: {REPORTS_DATA}')
print(f'FIGURES_DATA: {FIGURES_DATA}')

In [ ]:
plotly_config = {'responsive': True, 'displayModeBar': True, 'displaylogo': False, 'modeBarButtonsToRemove': ['lasso2d', 'select2d']}

def save_fig(fig, name: str):
    html_path = FIGURES_DATA / f'{name}.html'
    fig.write_html(html_path, config=plotly_config)
    try:
        png_path = FIGURES_DATA / f'{name}.png'
        fig.write_image(png_path, width=1200, height=600)
    except:
        pass
    return html_path

def plotly_table(df: pd.DataFrame, title: str = '', max_rows: int = 50):
    df_display = df.head(max_rows).copy()
    fig = go.Figure(data=[go.Table(
        header=dict(values=[f'<b>{col}</b>' for col in df_display.columns],
                    fill_color='#0077B6', align='left', font=dict(color='white', size=12)),
        cells=dict(values=[df_display[col] for col in df_display.columns],
                   fill_color='rgba(240,240,240,0.5)', align='left',
                   font=dict(size=11), height=25))])
    fig.update_layout(title=title, height=min(len(df_display)*30+100, 800),
                     margin=dict(l=0, r=0, t=40, b=0))
    return fig

print('Helpers loaded.')

## 2. Carregamento Consolidado dos Metadados

In [ ]:
def metadata_sort_key(path: Path) -> str:
    match = re.fullmatch(r'metadados(\d{6}|\d{8})\.json', path.name)
    return match.group(1) if match else path.name

metadata_files = sorted(METADATA_RAW.rglob('metadados*.json'), key=metadata_sort_key)
if not metadata_files:
    raise FileNotFoundError(f'Nenhum arquivo metadados*.json em {METADATA_RAW}')

print(f'Arquivos encontrados: {len(metadata_files)}')
print(f'Intervalo: {metadata_files[0].name} a {metadata_files[-1].name}')

In [ ]:
def load_metadata(path: Path) -> pd.DataFrame:
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    if isinstance(data, list):
        df = pd.DataFrame(data)
    elif isinstance(data, dict):
        list_vals = [v for v in data.values() if isinstance(v, list)]
        df = pd.DataFrame(list_vals[0]) if len(list_vals) == 1 else pd.json_normalize(data)
    else:
        raise ValueError(f'Formato JSON inesperado: {type(data)}')
    df['arquivo_origem'] = path.name
    return df

metadata_list = []
for path in metadata_files:
    try:
        metadata_list.append(load_metadata(path))
    except Exception as e:
        print(f'Erro em {path.name}: {e}')

df = pd.concat(metadata_list, ignore_index=True) if metadata_list else pd.DataFrame()
print(f'Metadados consolidados: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas')
print(f'Memória: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

## 3. Qualidade dos Dados

In [ ]:
missing_pct = (df.isna().sum() / len(df) * 100).sort_values()
missing_pct = missing_pct[missing_pct > 0]

fig = go.Figure(data=[go.Bar(y=missing_pct.index, x=missing_pct.values, orientation='h', marker_color='#E74C3C')])
fig.update_layout(title='Taxa de Valores Ausentes por Coluna (%)',
                 xaxis_title='% Ausente', yaxis_title='Coluna',
                 height=max(400, len(missing_pct)*25), showlegend=False)
save_fig(fig, '01_taxa_ausentes')
fig.show()

## 4. Evolução Temporal

In [ ]:
df['dataPublicacao'] = pd.to_datetime(df.get('dataPublicacao', pd.Series([pd.NaT]*len(df))), errors='coerce')
if 'dataPublicacao' in df.columns and df['dataPublicacao'].notna().any():
    print(f'Intervalo: {df["dataPublicacao"].min()} a {df["dataPublicacao"].max()}')
    df['ano'] = df['dataPublicacao'].dt.year
    df['mes'] = df['dataPublicacao'].dt.month

In [ ]:
# Volume diário com média móvel
if 'dataPublicacao' in df.columns and df['dataPublicacao'].notna().any():
    daily = df.groupby(df['dataPublicacao'].dt.date).size().reset_index(name='documentos')
    daily.columns = ['data', 'documentos']
    daily['data'] = pd.to_datetime(daily['data'])
    daily = daily.sort_values('data')
    daily['ma7'] = daily['documentos'].rolling(7, center=True).mean()
    daily['ma30'] = daily['documentos'].rolling(30, center=True).mean()
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=daily['data'], y=daily['documentos'], name='Diário', mode='lines', opacity=0.5, line=dict(color='#3498DB', width=1)))
    fig.add_trace(go.Scatter(x=daily['data'], y=daily['ma7'], name='Média Móvel 7d',
                            mode='lines', line=dict(color='#F39C12', width=2)))
    fig.update_layout(title='Volume Diário de Documentos',
                     xaxis_title='Data', yaxis_title='Documentos',
                     hovermode='x unified', height=500)
    save_fig(fig, '02_volume_diario')
    fig.show()

## 5. Composição Documental

In [ ]:
# Tipo de documento
if 'tipoDocumento' in df.columns:
    tipo_counts = df['tipoDocumento'].fillna('(vazio)').value_counts().head(15)
    fig = go.Figure(data=[go.Bar(y=tipo_counts.index, x=tipo_counts.values,
                                orientation='h', marker_color='#9B59B6')])
    fig.update_layout(title='Tipos de Documento Mais Frequentes',
                     xaxis_title='Documentos', yaxis_title='Tipo',
                     height=500, showlegend=False)
    save_fig(fig, '03_tipo_documento')
    fig.show()

## 6. Análise de Ministros

In [ ]:
col_ministro = 'ministro' if 'ministro' in df.columns else 'NM_MINISTRO'
if col_ministro in df.columns:
    ministro_counts = df[col_ministro].fillna('(vazio)').value_counts().head(20)
    fig = go.Figure(data=[go.Bar(y=ministro_counts.index, x=ministro_counts.values,
                               orientation='h', marker_color='#3498DB')])
    fig.update_layout(title='Top 20 Ministros', xaxis_title='Documentos',
                     yaxis_title='Ministro', height=500, showlegend=False)
    save_fig(fig, '04_ministros')
    fig.show()

## 7. Processos (CNJ)

In [ ]:
def parse_cnj_number(process_str: str) -> dict:
    if pd.isna(process_str):
        return {}
    process_str = str(process_str).strip().replace('.', '').replace('-', '')
    match = re.match(r'(\d{7})(\d{2})(\d{4})(\d)(\d{2})(\d{4})', process_str)
    if not match:
        return {}
    seq, check, year, segment, tribunal, origin = match.groups()
    segments = {'1': 'STF', '3': 'STJ', '4': 'JF', '5': 'JT', '8': 'JE'}
    return {'ano_cnj': int(year), 'segmento': segments.get(segment, segment)}

if 'processo' in df.columns:
    cnj_parsed = df['processo'].apply(parse_cnj_number).apply(pd.Series)
    df = pd.concat([df, cnj_parsed], axis=1)
    print(f'Processos parseados: {cnj_parsed.notna().sum().sum()} campos')

In [ ]:
# Distribuição por segmento
if 'segmento' in df.columns:
    seg_counts = df['segmento'].fillna('N/A').value_counts()
    fig = go.Figure(data=[go.Pie(labels=seg_counts.index, values=seg_counts.values)])
    fig.update_layout(title='Distribuição por Segmento de Origem')
    save_fig(fig, '05_segmento')
    fig.show()

## 8. Assuntos

In [ ]:
lookup_path = REFERENCE_DATA / 'assuntos' / 'processed' / 'assuntos_lookup.parquet'
if lookup_path.exists():
    assuntos_lookup = pd.read_parquet(lookup_path)
    print(f'Lookup carregado: {len(assuntos_lookup)} códigos')
else:
    assuntos_lookup = pd.DataFrame()
    print('Lookup não encontrado')

In [ ]:
# Top assuntos
if 'assuntos' in df.columns:
    assuntos_extracted = df['assuntos'].fillna('').astype(str).str.extract(r'(\d+)')[0]
    top_assuntos = assuntos_extracted.value_counts().head(15)
    fig = go.Figure(data=[go.Bar(y=top_assuntos.index, x=top_assuntos.values,
                               orientation='h', marker_color='#8E44AD')])
    fig.update_layout(title='Top 15 Códigos de Assunto',
                     xaxis_title='Documentos', height=500, showlegend=False)
    save_fig(fig, '06_assuntos')
    fig.show()

## 9. Síntese Executiva

In [ ]:
print(f'Total de documentos: {len(df):,}')
print(f'Intervalo temporal: {df["dataPublicacao"].min() if "dataPublicacao" in df.columns else "N/A"} a {df["dataPublicacao"].max() if "dataPublicacao" in df.columns else "N/A"}')
print(f'Colunas: {df.shape[1]}')
print(f'Memória: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')

figs = sorted((FIGURES_DATA).glob('*.html'))
print(f'\nGráficos gerados: {len(figs)}')
for fig in figs:
    print(f'  - {fig.name}')